## Deep MLP para MNIST por Jaime Ciriaco

En esta sección cargamos el dataset MNIST desde Keras, lo normalizamos a  [0, 1] y separamos un conjunto de validación a partir del conjunto de entrenamiento. 

Podríamos hacerlo con train_test_split pero preferimos cerrarnos al libro dejándolo de forma sencilla.

In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt

tf.random.set_seed(11)

(X_train_full, y_train_full), (X_test, y_test) = tf.keras.datasets.mnist.load_data()

X_train_full = X_train_full.astype("float32") / 255.
X_test = X_test.astype("float32") / 255.

X_valid = X_train_full[:5000]
y_valid = y_train_full[:5000]
X_train = X_train_full[5000:]
y_train = y_train_full[5000:]

X_train.shape, X_valid.shape, X_test.shape

Definimos una función generalista que construye un MLP con un número variable de capas ocultas y neuronas. Así podemos reutilizar el mismo código para todas las configuraciones que se demandan.

In [ ]:
def build_mlp(n_layers=3, n_neurons=128, input_shape=(28, 28)):
    model = tf.keras.Sequential()
    model.add(tf.keras.layers.Flatten(input_shape=input_shape))
    for _ in range(n_layers):
        model.add(tf.keras.layers.Dense(n_neurons, activation="relu"))
    model.add(tf.keras.layers.Dense(10, activation="softmax"))
    return model

Añadimos un callback que aumenta el learning rate exponencialmente en cada batch. Por ende, podremos representar la pérdida en función del learning rate y elegir un valor en base a los resultados.

In [ ]:
class ExponentialLRScheduler(tf.keras.callbacks.Callback):
    def __init__(self, initial_lr, factor, max_lr):
        super().__init__()
        self.initial_lr = initial_lr
        self.factor = factor
        self.max_lr = max_lr
        self.lrs = []
        self.losses = []

    def on_train_begin(self, logs=None):
        self.model.optimizer.learning_rate = self.initial_lr

    def on_batch_end(self, batch, logs=None):
        lr = tf.keras.backend.get_value(self.model.optimizer.learning_rate)
        self.lrs.append(lr)
        self.losses.append(logs["loss"])

        lr *= self.factor
        self.model.optimizer.learning_rate = lr

        if lr > self.max_lr:
            self.model.stop_training = True

Elegimos un learning rate inicial muy pequeño y uno final grande. El factor se calcula para que, en un número de pasos determinado, el LR vaya desde start_lr hasta end_lr tal que, en cada batch, el LR se multiplica por ese factor.

In [ ]:
batch_size = 128
steps_per_epoch = len(X_train) // batch_size

start_lr = 1e-5
end_lr = 1.0

total_steps = steps_per_epoch
factor = (end_lr / start_lr) ** (1 / total_steps)
factor

In [ ]:
tf.random.set_seed(11)

model_lr = build_mlp(n_layers=3, n_neurons=128)

optimizer = tf.keras.optimizers.SGD(learning_rate=start_lr, momentum=0.9)

model_lr.compile(loss="sparse_categorical_crossentropy",
                 optimizer=optimizer,
                 metrics=["accuracy"])

exp_lr_cb = ExponentialLRScheduler(initial_lr=start_lr,
                                   factor=factor,
                                   max_lr=end_lr)

history_lr = model_lr.fit(
    X_train, y_train,
    epochs=1,
    batch_size=batch_size,
    callbacks=[exp_lr_cb],
    verbose=1
)

Representamos la pérdida en función del learning rate en escala logarítmica con plt. El LR correcto suele estar justo antes de que la pérdida empiece a dispararse (eso si pusieramos algún valor máximo de LR superior a 1), en este caso, descender su training loss.

In [ ]:
lrs = exp_lr_cb.lrs
losses = exp_lr_cb.losses

plt.figure(figsize=(8, 5))
plt.plot(lrs, losses)
plt.xscale("log")
plt.xlabel("Learning rate (escala logarítmica)")
plt.ylabel("Training loss")
plt.grid(True)
plt.show()

A partir de la gráfica observamos que la pérdida comienza a disminuir de forma clara a partir de 10^-3, y empieza a crecer bruscamente alrededor de 0.00. Por tanto se elige un learning rate de 0.02, ligeramente por debajo del punto donde la pérdida empieza a aumentar.

In [ ]:
best_lr = 0.02
best_lr

Definimos callbacks de EarlyStopping para evitar sobreajuste y ModelCheckpoint para guardar el mejor modelo durante el entrenamiento.

In [ ]:
early_stopping_cb = tf.keras.callbacks.EarlyStopping(
    patience=5,
    restore_best_weights=True,
    monitor="val_loss"
)

checkpoint_cb = tf.keras.callbacks.ModelCheckpoint(
    "best_mnist_model.keras",
    save_best_only=True,
    monitor="val_loss"
)

In [ ]:
from pathlib import Path
from time import strftime

def get_run_logdir(root_logdir="logs_mnist"):
    return Path(root_logdir) / strftime("run_%Y_%m_%d_%H_%M_%S")

log_dir = str(get_run_logdir())
tensorboard_cb = tf.keras.callbacks.TensorBoard(log_dir=log_dir)

En esta sección comparamos cuatro arquitecturas:

    3 capas ocultas con 64 neuronas

    3 capas ocultas con 256 neuronas

    4 capas ocultas con 64 neuronas

    4 capas ocultas con 256 neuronas

Todas usan el mismo learning rate obtenido en la búsqueda anterior.

In [ ]:
configs = [
    {"name": "3x64",  "n_hidden": 3, "n_neurons": 64},
    {"name": "3x256", "n_hidden": 3, "n_neurons": 256},
    {"name": "4x64",  "n_hidden": 4, "n_neurons": 64},
    {"name": "4x256", "n_hidden": 4, "n_neurons": 256},
]

results = {}

for cfg in configs:
    print("\n Entrenando modelo:", cfg["name"])
    tf.random.set_seed(42)  # para reproducibilidad
    
    model = build_mlp(n_layers=cfg["n_hidden"],
                      n_neurons=cfg["n_neurons"])
    
    optimizer = tf.keras.optimizers.SGD(learning_rate=best_lr, momentum=0.9)
    model.compile(loss="sparse_categorical_crossentropy",
                  optimizer=optimizer,
                  metrics=["accuracy"])

    history = model.fit(
        X_train, y_train,
        epochs=50,
        batch_size=128,
        validation_data=(X_valid, y_valid),
        callbacks=[early_stopping_cb, checkpoint_cb],
        verbose=0
    )

    test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
    
    results[cfg["name"]] = {
        "history": history.history,
        "test_loss": test_loss,
        "test_acc": test_acc
    }
    
    print(f"Modelo {cfg['name']} - Test accuracy: {test_acc:.4f}")

In [ ]:
plt.figure(figsize=(8, 5))

for cfg in configs:
    name = cfg["name"]
    history = results[name]["history"]
    plt.plot(history["val_accuracy"], label=name)

plt.xlabel("Epoch")
plt.ylabel("Validation accuracy")
plt.legend()
plt.grid(True)
plt.show()

En nuestro caso EarlyStopping monitoriza val_loss, por lo que el modelo se queda con los pesos del epoch con menor pérdida de validación, que no coincide exactamente con el epoch donde la val_accuracy es máxima, aunque ambos son muy similares. Por eso aunque veamos que en el modelo 3x256 tiene valores de epoch que pasan del 98.% de accuracy, no es éste el porcentaje que sacamos.

In [ ]:
best_3_name = None
best_4_name = None
best_3_acc = -np.inf
best_4_acc = -np.inf

for name, res in results.items():
    test_acc = res["test_acc"]
    if name.startswith("3x") and test_acc > best_3_acc:
        best_3_acc = test_acc
        best_3_name = name
    if name.startswith("4x") and test_acc > best_4_acc:
        best_4_acc = test_acc
        best_4_name = name

best_3_name, best_3_acc, best_4_name, best_4_acc

A continuación reentrenamos las arquitecturas ganadoras de 3 y 4 capas con un learning rate diferente (en este caso más pequeño) y comparamos sus resultados con los de la parte anterior.

In [ ]:
new_lr = best_lr / 3

def train_with_lr(n_layers, n_neurons, lr, name_suffix):
    tf.random.set_seed(42)
    model = build_mlp(n_layers=n_layers, n_neurons=n_neurons)
    optimizer = tf.keras.optimizers.SGD(learning_rate=lr, momentum=0.9)
    model.compile(loss="sparse_categorical_crossentropy",
                  optimizer=optimizer,
                  metrics=["accuracy"])
    
    history = model.fit(
        X_train, y_train,
        epochs=50,
        batch_size=128,
        validation_data=(X_valid, y_valid),
        callbacks=[early_stopping_cb],
        verbose=0
    )
    
    test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
    print(f"{name_suffix} - LR={lr:.5f} - Test accuracy: {test_acc:.4f}")
    return history.history, test_loss, test_acc


def parse_name(name):
    parts = name.split("x")
    return int(parts[0]), int(parts[1])

n_hidden_3, n_neurons_3 = parse_name(best_3_name)
n_hidden_4, n_neurons_4 = parse_name(best_4_name)

history_3_new, loss_3_new, acc_3_new = train_with_lr(
    n_hidden_3, n_neurons_3, new_lr, name_suffix=f"{best_3_name}_newLR"
)

history_4_new, loss_4_new, acc_4_new = train_with_lr(
    n_hidden_4, n_neurons_4, new_lr, name_suffix=f"{best_4_name}_newLR"
)

In [ ]:
old_hist_3 = results[best_3_name]["history"]

plt.figure(figsize=(8, 5))
plt.plot(old_hist_3["val_accuracy"], label=f"{best_3_name} - val_acc (LR original)")
plt.plot(history_3_new["val_accuracy"], label=f"{best_3_name} - val_acc (LR nuevo)")
plt.xlabel("Epoch")
plt.ylabel("Validation accuracy")
plt.legend()
plt.grid(True)
plt.show()

---

#### Resumén

Mediante la búsqueda de learning rate seleccionamos un valor cercano a 0 (0.02), que permitió una convergencia rápida sin inestabilizar el entrenamiento.

Las redes con 3 y 4 capas ocultas alcanzan precisiones similares, aunque la arquitectura 3×256 ofreció la mejor val_accuracy.

Aumentar el número de neuronas mejora la capacidad del modelo, pero también incrementa el riesgo de sobreajuste (sesgo); el uso de early stopping lo ha controlado.

Al reducir el learning rate a 0.02/3 la convergencia fue más lenta y además la precisión bajó, lo que sugiere que el LR inicial estaba cerca de un valor mejor como demostró el graficado inicial.

Un LR demasiado pequeño hace que el entrenamiento sea más lento y puede dejar al modelo atrapado en soluciones peores